<a href="https://colab.research.google.com/github/Harmain-Kanwal/Flyrank-AI/blob/main/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

I built the ranked queue as a transparent, rule-based priority score rather than a classifier output — Weeks 5–6's model comparisons didn't hold up on inspection (see the code comments below for what was wrong and how it's fixed here), so this playbook stands on directly observed signals instead: decline trend, staleness, visibility, position, and content depth.

Six archetypes come out of this: **Refresh Content** (declining + stale + already visible), **Optimize for Position** (sitting in striking distance with real traffic behind it), **Rewrite Title/Meta** (visible but under-clicked relative to peers), **Consolidate or Retire** (old, declining, barely seen), **Expand Thin Content**, and **Monitor Only** (no action). Exact counts are printed below.

**The decay/refresh insight:** thin content in this portfolio almost never earns real visibility regardless of age or intent — the vast majority of sub-1,000-word pages sit in the lowest impression tier. So "write more content" is a much smaller lever here than "fix what's already getting seen but decaying or under-converting." Budget should skew toward Refresh and Optimize, not new writing.

In [ ]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# ----------------------------------------------------------------------
# Label fix: Week 6 used `.str.contains("declin")` against trend_direction,
# whose real values are down/stable/new/up/flat -- none contain "declin",
# so that column was 0 for every row. The correct rule (per the data
# dictionary) is trend_direction == "down".
# ----------------------------------------------------------------------
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# avg_position == 0 means "no position data," not rank zero -- never treat
# it as a good position.
df["has_position_data"] = (df["avg_position"] > 0).astype(int)

# CTR is heavily zero-skewed (median 0.07%), so a fixed "< 2%" or "< 0.02%"
# cutoff either flags almost everyone or almost no one. Rank-based instead.
df["ctr_pctile"] = df["ctr"].rank(pct=True)

VIS_TIERS = ["moderate", "good", "excellent"]  # real, measured traffic behind the page


def reason_codes(row):
    codes = []
    if row["is_declining"] == 1:
        codes.append("DECLINING_TREND")
    if row["age_tier"] in ["181-365", "365+"] or row["freshness_tier"] in ["91-180", "181+"]:
        codes.append("STALE_CONTENT")
    if row["impression_tier"] == "low":
        codes.append("LOW_VISIBILITY")
    if row["impression_tier"] in VIS_TIERS:
        codes.append("HIGH_IMPRESSIONS")
    if row["position_tier"] == "striking":
        codes.append("STRIKING_DISTANCE")
    if row["word_count_tier"] == "<1000":
        codes.append("THIN_CONTENT")
    if row["ctr_pctile"] <= 0.25:
        codes.append("LOW_CTR")
    if len(codes) >= 3:
        codes.append("REFRESH_PRIORITY")
    return ", ".join(codes)


def assign_archetype(row):
    # Priority order matters -- first match wins, so the most urgent/cheapest wins ties.
    if row["age_tier"] == "365+" and row["impression_tier"] == "low" and row["is_declining"] == 1:
        return "RETIRE_OR_CONSOLIDATE"
    if row["is_declining"] == 1 and row["impression_tier"] in VIS_TIERS and (
        row["age_tier"] in ["181-365", "365+"] or row["freshness_tier"] in ["91-180", "181+"]
    ):
        return "REFRESH_CONTENT"
    # Gated on real visibility on purpose: the data dictionary warns position_tier
    # medians need a volume floor stated, so "striking distance" alone isn't enough.
    if row["position_tier"] == "striking" and row["impression_tier"] in VIS_TIERS:
        return "OPTIMIZE_FOR_POSITION"
    if row["word_count_tier"] == "<1000" and row["impression_tier"] in VIS_TIERS:
        return "EXPAND_CONTENT"
    if row["ctr_pctile"] <= 0.25 and row["impression_tier"] in VIS_TIERS and row["is_declining"] == 0:
        return "IMPROVE_TITLE_METADATA"
    return "MONITOR_ONLY"


ACTION_MAP = {
    "RETIRE_OR_CONSOLIDATE": "Consolidate or retire",
    "REFRESH_CONTENT": "Refresh / update content",
    "OPTIMIZE_FOR_POSITION": "On-page optimization (title, internal links, schema)",
    "EXPAND_CONTENT": "Expand thin content",
    "IMPROVE_TITLE_METADATA": "Rewrite title/meta for CTR",
    "MONITOR_ONLY": "Monitor only -- no action",
}
PRIORITY_TIER = {
    "RETIRE_OR_CONSOLIDATE": 1, "REFRESH_CONTENT": 2, "OPTIMIZE_FOR_POSITION": 3,
    "EXPAND_CONTENT": 4, "IMPROVE_TITLE_METADATA": 5, "MONITOR_ONLY": 6,
}

df["archetype"] = df.apply(assign_archetype, axis=1)
df["reason_code"] = df.apply(reason_codes, axis=1)
df["action"] = df["archetype"].map(ACTION_MAP)
df["priority_tier"] = df["archetype"].map(PRIORITY_TIER)

# Rank within priority tier by traffic at stake -- bigger audience, bigger payoff.
queue = df[df["archetype"] != "MONITOR_ONLY"].sort_values(
    by=["priority_tier", "impressions_90d"], ascending=[True, False]
).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

print(df["archetype"].value_counts())
print()
print(df["archetype"].value_counts(normalize=True).round(3))
print()
print("Total pages flagged for some action:", len(queue), f"({len(queue)/len(df):.1%})")

preview_cols = ["rank", "archetype", "action", "reason_code",
                "impressions_90d", "content_age_days", "avg_position", "trend_pct"]
queue[preview_cols].head(10)

archetype
MONITOR_ONLY              17822
REFRESH_CONTENT            7240
OPTIMIZE_FOR_POSITION      3054
IMPROVE_TITLE_METADATA     1118
RETIRE_OR_CONSOLIDATE       755
EXPAND_CONTENT               11
Name: count, dtype: int64

archetype
MONITOR_ONLY              0.594
REFRESH_CONTENT           0.241
OPTIMIZE_FOR_POSITION     0.102
IMPROVE_TITLE_METADATA    0.037
RETIRE_OR_CONSOLIDATE     0.025
EXPAND_CONTENT            0.000
Name: proportion, dtype: float64

Total pages flagged for some action: 12178 (40.6%)


,rank,archetype,action,reason_code,impressions_90d,content_age_days,avg_position,trend_pct
0,1,RETIRE_OR_CONSOLIDATE,Consolidate or retire,"DECLINING_TREND, STALE_CONTENT, LOW_VISIBILITY...",298,517,71.4,-97.9
1,2,RETIRE_OR_CONSOLIDATE,Consolidate or retire,"DECLINING_TREND, STALE_CONTENT, LOW_VISIBILITY...",297,502,13.9,-100.0
2,3,RETIRE_OR_CONSOLIDATE,Consolidate or retire,"DECLINING_TREND, STALE_CONTENT, LOW_VISIBILITY...",295,460,18.2,-27.3
3,4,RETIRE_OR_CONSOLIDATE,Consolidate or retire,"DECLINING_TREND, STALE_CONTENT, LOW_VISIBILITY...",293,557,12.2,-93.4
4,5,RETIRE_OR_CONSOLIDATE,Consolidate or retire,"DECLINING_TREND, STALE_CONTENT, LOW_VISIBILITY...",293,502,58.2,-74.2
5,6,RETIRE_OR_CONSOLIDATE,Consolidate or retire,"DECLINING_TREND, STALE_CONTENT, LOW_VISIBILITY...",292,390,33.7,-85.7
6,7,RETIRE_OR_CONSOLIDATE,Consolidate or retire,"DECLINING_TREND, STALE_CONTENT, LOW_VISIBILITY...",292,480,49.2,-37.0
7,8,RETIRE_OR_CONSOLIDATE,Consolidate or retire,"DECLINING_TREND, STALE_CONTENT, LOW_VISIBILITY...",292,445,40.2,-30.8
8,9,RETIRE_OR_CONSOLIDATE,Consolidate or retire,"DECLINING_TREND, STALE_CONTENT, LOW_VISIBILITY...",291,460,28.3,-27.3
9,10,RETIRE_OR_CONSOLIDATE,Consolidate or retire,"DECLINING_TREND, STALE_CONTENT, LOW_VISIBILITY...",291,463,58.2,-60.1


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this, for what:** a content/SEO team lead deciding where to spend a sprint's or quarter's editorial effort across these 32 clients — a triage tool, not an execution engine.

**Where it stops being valid:**
- Every row in this dataset has `content_age_days >= 90` — the playbook has never seen content younger than 90 days, so it shouldn't be applied to brand-new pages.
- It only covers the 32 clients in this slice. A new client's traffic patterns, verticals, or tracking setup aren't represented.
- `is_declining` is a 30-day-vs-prior-30-day impression comparison. It flags a pattern, not a cause — it can't distinguish a real content problem from a tracking outage, a seasonal dip, or a SERP layout change.
- It doesn't know about competitor moves, algorithm updates, brand/compliance value of a page, or planned migrations.
- `position_tier == "striking"` is gated on measured visibility (`impression_tier` in moderate/good/excellent) specifically because tier medians can look better than they are on thin volume — this reduces that risk at the portfolio level but doesn't eliminate it on any individual page.

In [ ]:
# Numbers backing the limits above.
print("content_age_days range:", df["content_age_days"].min(), "-", df["content_age_days"].max())
print("clients covered:", df["client_id"].nunique())
print("rows with avg_position == 0 (no position data, excluded from position logic):",
      (df["avg_position"] == 0).sum())
print("rows where impressions_prev_30d was 0 (trend_pct filled from a blank):",
      (df["impressions_prev_30d"] == 0).sum(), "-- treat trend_pct near 0 for these with care")

content_age_days range: 90 - 564
clients covered: 32
rows with avg_position == 0 (no position data, excluded from position logic): 1205
rows where impressions_prev_30d was 0 (trend_pct filled from a blank): 3388 -- treat trend_pct near 0 for these with care


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**What a person must check before acting on any row:**
- The page still exists and hasn't already been updated more recently than this export.
- For Refresh/Optimize/Retire: read the actual page. Does the drop look like real content decay, or a known external cause (seasonality, a SERP feature, an algorithm update, a migration)?
- For Optimize for Position: confirm the keyword/intent it's ranking for is still the right one to target — a page can sit in striking distance for a query that no longer matters to the business.
- For Retire/Consolidate: check backlinks, internal links, and whether the page serves a non-traffic purpose (legal, brand, reference) before removing or merging it.
- Cost/value sanity check: does the editorial cost of the recommended action (writer hours) look proportional to the traffic at stake (`impressions_90d`, `search_volume`)? Refresh and Optimize are comparatively cheap and target already-visible traffic — the best ROI in this queue. Expand is the most expensive per page and, per Section 1's finding, targets the smallest real opportunity here.

**What must NOT be automated:**
- No auto-publishing or auto-editing of copy — the queue flags intent, a person writes the words.
- No auto-retiring or auto-unpublishing a page without sign-off, especially since `RETIRE_OR_CONSOLIDATE` can catch pages that are seasonal or serve a non-traffic purpose.
- No use of this queue for performance reviews or staffing decisions — it scores pages, not the people who wrote them.
- No automated action on YMYL content (health, finance, legal) without subject-matter or compliance review — this rule set has no idea what regulatory risk looks like.
- No causal language, and no automated action based on `is_declining` alone treated as "why" — it's an observed 30-day comparison, not a diagnosis.
- No application to clients or content outside this validated slice (see Section 2) without re-checking these thresholds hold there.

In [ ]:
# A few borderline rows worth a human's second look before anything downstream trusts them.
edge_cases = df[(df["avg_position"] == 0) & (df["archetype"] != "MONITOR_ONLY")]
print("Flagged pages with no position data at all (verify before acting):", len(edge_cases))

extreme_drop = queue[queue["trend_pct"] <= -95].head(5)
print()
print("Pages with an extreme (>95%) impression drop -- check for a tracking break before treating as content decay:")
extreme_drop[["rank", "archetype", "impressions_90d", "trend_pct"]]

Flagged pages with no position data at all (verify before acting): 0

Pages with an extreme (>95%) impression drop -- check for a tracking break before treating as content decay:


,rank,archetype,impressions_90d,trend_pct
0,1,RETIRE_OR_CONSOLIDATE,298,-97.9
1,2,RETIRE_OR_CONSOLIDATE,297,-100.0
40,41,RETIRE_OR_CONSOLIDATE,264,-95.3
50,51,RETIRE_OR_CONSOLIDATE,260,-100.0
52,53,RETIRE_OR_CONSOLIDATE,259,-100.0


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This is a rules-based playbook, not a trained model, so there's no "retrain" in the ML sense — the equivalent trigger is **recalibration**. Watch for:
- **Archetype drift:** if the share of pages in any archetype moves more than ~10 percentage points from this run's baseline (below) without a known cause, check the pipeline before trusting the queue — a sudden jump usually means a tracking break, not a sudden wave of real content decay.
- **Volume floor breaks:** if the `impression_tier` distribution shifts (e.g., a client's traffic collapses), the `moderate/good/excellent` gates this playbook relies on may stop meaning what they meant here — re-check before reusing the same tier cutoffs.
- **New client onboarding:** a new client's first 90 days of content won't have valid `content_age_days`/tier values yet (see Section 2) — exclude until they clear that window.
- **Schedule:** re-run and compare quarterly at minimum, or immediately after any known tracking/pipeline change (a GA4 or GSC config change, a client migration).

In [ ]:
import json

baseline_snapshot = {
    "run_date": pd.Timestamp.today().strftime("%Y-%m-%d"),
    "archetype_share": df["archetype"].value_counts(normalize=True).round(4).to_dict(),
}
print(json.dumps(baseline_snapshot, indent=2))


def check_drift(current_share: dict, baseline_share: dict, threshold: float = 0.10) -> list:
    """Flags archetypes whose share moved more than `threshold` from baseline."""
    alerts = []
    for archetype, base_pct in baseline_share.items():
        cur_pct = current_share.get(archetype, 0)
        if abs(cur_pct - base_pct) > threshold:
            alerts.append(f"{archetype}: {base_pct:.1%} -> {cur_pct:.1%}")
    return alerts

# Example call once a future run exists:
# check_drift(new_run_share, baseline_snapshot["archetype_share"])

{
  "run_date": "2026-08-30",
  "archetype_share": {
    "MONITOR_ONLY": 0.5941,
    "REFRESH_CONTENT": 0.2413,
    "OPTIMIZE_FOR_POSITION": 0.1018,
    "IMPROVE_TITLE_METADATA": 0.0373,
    "RETIRE_OR_CONSOLIDATE": 0.0252,
    "EXPAND_CONTENT": 0.0004
  }
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exporting the ranked queue (CSV, regenerated each run, intentionally not committed — the CI leak-guard blocks data files under `work/`), the archetype chart (committed, `work/figures/`), and the run metrics (committed, `work/outputs/playbook_metrics.json`) — these are what next week's paper cites back to.

In [ ]:
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

export_cols = ["rank", "archetype", "action", "reason_code", "content_id", "client_id",
               "priority_tier", "impressions_90d", "search_volume", "content_age_days",
               "days_since_last_update", "avg_position", "position_tier", "ctr",
               "trend_pct", "trend_direction", "word_count_tier"]

queue[export_cols].to_csv("work/outputs/content_action_playbook.csv", index=False)

counts = df["archetype"].value_counts().reindex(
    ["MONITOR_ONLY", "REFRESH_CONTENT", "OPTIMIZE_FOR_POSITION",
     "IMPROVE_TITLE_METADATA", "RETIRE_OR_CONSOLIDATE", "EXPAND_CONTENT"]
)
plt.figure(figsize=(8, 5))
counts.plot(kind="barh")
plt.xlabel("Number of pages")
plt.title("Content action playbook -- pages per archetype")
plt.tight_layout()
plt.savefig("work/figures/archetype_counts.png", dpi=150)
plt.show()

metrics = {
    "dataset": "content_refresh_anonymized.csv",
    "n_rows": int(len(df)),
    "n_clients": int(df["client_id"].nunique()),
    "n_flagged_for_action": int(len(queue)),
    "pct_flagged_for_action": round(len(queue) / len(df), 4),
    "archetype_counts": {k: int(v) for k, v in counts.items()},
    "thresholds": {
        "low_ctr_definition": "bottom 25th percentile of ctr, rank-based -- ctr is heavily zero-skewed",
        "high_visibility_definition": "impression_tier in [moderate, good, excellent]",
        "stale_definition": "age_tier in [181-365, 365+] OR freshness_tier in [91-180, 181+]",
        "striking_distance_definition": "position_tier == striking, gated on measured visibility",
    },
    "known_limits": [
        "validated only on this 30,000-row / 32-client slice; content_age_days >= 90 for every row",
        "is_declining reflects a 30-day-vs-prior-30-day impression comparison, not a diagnosed cause",
    ],
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported:")
print(" - work/outputs/content_action_playbook.csv (gitignored, regenerate on demand)")
print(" - work/outputs/playbook_metrics.json (commit this)")
print(" - work/figures/archetype_counts.png (commit this)")

Exported:
 - work/outputs/content_action_playbook.csv (gitignored, regenerate on demand)
 - work/outputs/playbook_metrics.json (commit this)
 - work/figures/archetype_counts.png (commit this)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.